In [1]:
!pip install -q pandas pyarrow

In [2]:
from pathlib import Path
import pandas as pd

In [3]:
BASE_DIR = Path.cwd()
SHARED_DATA_DIR = (BASE_DIR / "../data/shared_data").resolve()

STAGING_DIR = SHARED_DATA_DIR / "staging"
MASTER_DIR = SHARED_DATA_DIR / "master"
LOGS_DIR = SHARED_DATA_DIR / "logs"

MASTER_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

VIDEOS_MASTER_FILE = MASTER_DIR / "videos_master.parquet"
COMMENTS_MASTER_FILE = MASTER_DIR / "comments_master.parquet"
RUNS_MASTER_FILE = LOGS_DIR / "runs_log_master.parquet"

In [4]:
def load_parquet_if_exists(path: Path) -> pd.DataFrame:
    if path.exists() and path.stat().st_size > 0:
        return pd.read_parquet(path)
    return pd.DataFrame()

def deduplicate_videos(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df.copy()
    x = df.copy()
    x["video_published_at"] = pd.to_datetime(x["video_published_at"], utc=True, errors="coerce")
    x["fetched_at_utc"] = pd.to_datetime(x["fetched_at_utc"], utc=True, errors="coerce")
    x = x.sort_values(["video_published_at", "fetched_at_utc"], ascending=[False, False], na_position="last")
    x = x.drop_duplicates(subset=["video_id"], keep="first")
    return x.reset_index(drop=True)

def deduplicate_comments(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df.copy()
    x = df.copy()
    x["published_at"] = pd.to_datetime(x["published_at"], utc=True, errors="coerce")
    x["updated_at"] = pd.to_datetime(x["updated_at"], utc=True, errors="coerce")
    x["fetched_at_utc"] = pd.to_datetime(x["fetched_at_utc"], utc=True, errors="coerce")
    x = x.sort_values(["updated_at", "published_at", "fetched_at_utc"], ascending=[False, False, False], na_position="last")
    x = x.drop_duplicates(subset=["comment_id"], keep="first")
    return x.reset_index(drop=True)

def merge_master_with_new(master: pd.DataFrame, new_df: pd.DataFrame, kind: str) -> pd.DataFrame:
    if master.empty and new_df.empty:
        return pd.DataFrame()

    if master.empty:
        combined = new_df.copy()
    elif new_df.empty:
        combined = master.copy()
    else:
        combined = pd.concat([master, new_df], ignore_index=True)

    if kind == "videos":
        return deduplicate_videos(combined)
    elif kind == "comments":
        return deduplicate_comments(combined)
    else:
        return combined.drop_duplicates().reset_index(drop=True)

In [5]:
video_files = sorted(STAGING_DIR.glob("*_videos_*.parquet"))
comment_files = sorted(STAGING_DIR.glob("*_comments_*.parquet"))
run_files = sorted(STAGING_DIR.glob("*_runs_*.parquet"))

print("Video staging files:", len(video_files))
print("Comment staging files:", len(comment_files))
print("Run log staging files:", len(run_files))

for f in video_files[:6]:
    print("VIDEO:", f.name)

for f in comment_files[:6]:
    print("COMMENT:", f.name)

for f in run_files[:6]:
    print("RUN:", f.name)

Video staging files: 6
Comment staging files: 6
Run log staging files: 6
VIDEO: leader_videos_20260410_125642.parquet
VIDEO: member_1_videos_20260410_130516.parquet
VIDEO: member_2_videos_20260410_131457.parquet
VIDEO: member_3_videos_20260410_134154.parquet
VIDEO: member_4_videos_20260410_134645.parquet
VIDEO: member_5_videos_20260410_135216.parquet
COMMENT: leader_comments_20260410_125642.parquet
COMMENT: member_1_comments_20260410_130516.parquet
COMMENT: member_2_comments_20260410_131457.parquet
COMMENT: member_3_comments_20260410_134154.parquet
COMMENT: member_4_comments_20260410_134645.parquet
COMMENT: member_5_comments_20260410_135216.parquet
RUN: leader_runs_20260410_125642.parquet
RUN: member_1_runs_20260410_130516.parquet
RUN: member_2_runs_20260410_131457.parquet
RUN: member_3_runs_20260410_134154.parquet
RUN: member_4_runs_20260410_134645.parquet
RUN: member_5_runs_20260410_135216.parquet


In [6]:
video_dfs = [pd.read_parquet(f) for f in video_files] if video_files else []
comment_dfs = [pd.read_parquet(f) for f in comment_files] if comment_files else []
run_dfs = [pd.read_parquet(f) for f in run_files] if run_files else []

staging_videos = pd.concat(video_dfs, ignore_index=True) if video_dfs else pd.DataFrame()
staging_comments = pd.concat(comment_dfs, ignore_index=True) if comment_dfs else pd.DataFrame()
staging_runs = pd.concat(run_dfs, ignore_index=True) if run_dfs else pd.DataFrame()

print("Staging videos rows:", 0 if staging_videos.empty else len(staging_videos))
print("Staging comments rows:", 0 if staging_comments.empty else len(staging_comments))
print("Staging runs rows:", 0 if staging_runs.empty else len(staging_runs))

Staging videos rows: 7504
Staging comments rows: 66509
Staging runs rows: 189


In [7]:
videos_master_old = load_parquet_if_exists(VIDEOS_MASTER_FILE)
comments_master_old = load_parquet_if_exists(COMMENTS_MASTER_FILE)
runs_master_old = load_parquet_if_exists(RUNS_MASTER_FILE)

print("Existing master videos:", 0 if videos_master_old.empty else len(videos_master_old))
print("Existing master comments:", 0 if comments_master_old.empty else len(comments_master_old))
print("Existing master runs:", 0 if runs_master_old.empty else len(runs_master_old))

Existing master videos: 7082
Existing master comments: 78413
Existing master runs: 427


In [8]:
videos_master_new = merge_master_with_new(videos_master_old, staging_videos, kind="videos")
comments_master_new = merge_master_with_new(comments_master_old, staging_comments, kind="comments")

if runs_master_old.empty and staging_runs.empty:
    runs_master_new = pd.DataFrame()
elif runs_master_old.empty:
    runs_master_new = staging_runs.drop_duplicates().reset_index(drop=True)
elif staging_runs.empty:
    runs_master_new = runs_master_old.drop_duplicates().reset_index(drop=True)
else:
    runs_master_new = pd.concat([runs_master_old, staging_runs], ignore_index=True).drop_duplicates().reset_index(drop=True)

print("Merged master videos:", 0 if videos_master_new.empty else len(videos_master_new))
print("Merged master comments:", 0 if comments_master_new.empty else len(comments_master_new))
print("Merged master runs:", 0 if runs_master_new.empty else len(runs_master_new))

Merged master videos: 13206
Merged master comments: 133110
Merged master runs: 616


In [9]:
if not videos_master_new.empty:
    print("Unique video_id:", videos_master_new["video_id"].nunique(), " / rows:", len(videos_master_new))

if not comments_master_new.empty:
    print("Unique comment_id:", comments_master_new["comment_id"].nunique(), " / rows:", len(comments_master_new))

Unique video_id: 13206  / rows: 13206
Unique comment_id: 133110  / rows: 133110


In [14]:
display(videos_master_new)
display(comments_master_new)
display(runs_master_new)

,video_id,channel_id,channel_title,title,description,tags,category_id,default_language,default_audio_language,video_published_at,duration,view_count,like_count,comment_count,search_keyword,fetched_at_utc,fetched_by
0,dwMneDhiPSM,UCLQNd80MchotmAF4nFtQblw,Guys Favorite Finds,REVIEW MAGEGEE SKY75 CREAMY KEYBOARD AND DIVOO...,⬇️ PRODUCTS IN THE VIDEO ⬇️\n-----------------...,,22,en,en,2026-04-10 05:39:09+00:00,PT3M18S,0,0.0,0.0,keyboard review,2026-04-10 05:43:28+00:00,member_3
1,MNNXaFtsYwo,UCfmSjwDkU88vTkSWaHajXGw,I & U Creation,Meesho jewellery 🩷 #meesho #shorts #jewellery,Meesho jewellery 🩷 #meesho #shorts #jewellery ...,meesho | meesho app | meesho finds | meesho lo...,26,en,en,2026-04-10 05:34:19+00:00,PT23S,326,1.0,0.0,jewelry unboxing,2026-04-10 06:00:57+00:00,member_5
2,5QPhf7jL3ZA,UCWEg9A7nns8tbbsnVE2e3UQ,MeeManuNaresh_vlogs,Meesho🌸 earrings 🤌 #shortsfeed #foryou #trendi...,Meesho🌸 earrings 🤌 #shortsfeed #foryou #trendi...,shorts | ytshorts | meesho | meesho Oxidized E...,24,en,te,2026-04-10 05:33:49+00:00,PT23S,143,1.0,0.0,jewelry review,2026-04-10 06:01:15+00:00,member_5
3,3Gk-6CYFhUc,UCBWt5StSsGB8zHz8MPNFHwg,BALLTMAX,The Coldest Moissanite Prong Cuban Link! Bling...,Full Review Video Coming Soon! @blingproud \n\...,Best Cuban Link | Best Moissanite Jewelry | Be...,22,en,en,2026-04-10 05:33:24+00:00,PT45S,965,10.0,1.0,jewelry review,2026-04-10 06:01:15+00:00,member_5
4,VCjQCO5ezhg,UCVWEvjL-YsEpEHRxbATMHcQ,Daquon the Gamer,Mario Fitness Unboxing & Using Your Gear! #ma...,More Information Below ⬇️ (Shopping Links. Des...,,24,en-US,en-US,2026-04-10 05:30:13+00:00,PT49S,2228,6.0,0.0,gaming accessories unboxing,2026-04-10 05:45:08+00:00,member_3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13201,wK2w4GqrBHI,UC2gTRhRGlo7jM1qyakfSCeA,AUTOO_360_INDIA,Windshield Par Dust & Scratches Se Bachao! Car...,🚗✨ Windshield Cleaning Made Powerful!\n\nAaj m...,,22,en,hi,2026-03-03 14:32:06+00:00,PT36S,939,19.0,0.0,tablet scratched easily,2026-04-02 13:10:43+00:00,member_1
13202,RietcsAUQ_k,UCWMYgBdapM6vGcwNXMMNd8A,ROAM it RALPH,I Built the Perfect Action Camera Setup for Tr...,Transforming the DJI Osmo Action 6 into a trad...,dji osmo action | telesin streetgrip | telesin...,26,en,en-US,2026-03-03 14:00:39+00:00,PT7M7S,645,9.0,3.0,camera lens accessories setup,2026-04-02 13:55:58+00:00,member_2
13203,krKC_xLyFGU,UCqW5ZNZUmWQ5ATgEf27JeMg,JAIKI TECH,DJI Nio 2 Drone Unboxing 😍 | 3 Battery Combo P...,🚁 DJI Nio 2 Drone Unboxing | 3 Battery Combo P...,dji neo 2 drone | dji mini drone | mini drone ...,28,hi,hi,2026-03-03 14:00:00+00:00,PT13M4S,27153,832.0,120.0,drone battery accessories setup,2026-04-02 13:55:12+00:00,member_2
13204,2gYK9v9fFUU,UCHd2QG5MHD0MTEkW1zGolbg,AnnaiChild and skinClinic,👂🌍 World Hearing Day – Protect Little Ears Tod...,👂🌍 World Hearing Day – Protect Little Ears Tod...,,26,en,ta,2026-03-03 13:42:23+00:00,PT7S,149,5.0,0.0,how to protect headphones,2026-04-02 13:12:06+00:00,member_1


,comment_id,video_id,thread_id,parent_comment_id,is_reply,author_channel_id,author_display_name,text_original,text_display,like_count,published_at,updated_at,total_reply_count,search_keyword,fetched_at_utc,fetched_by
0,UgwApYrk9BhCbRRFFzd4AaABAg,31i5MdossTo,UgwApYrk9BhCbRRFFzd4AaABAg,None,False,UCnVP89d_Qu6osTmqbmbaCJg,@stephdobbs-t3f,Cute❤❤❤❤❤❤,Cute❤❤❤❤❤❤,0,2026-04-10 06:03:09+00:00,2026-04-10 06:03:09+00:00,0.0,figurines unboxing,2026-04-10 06:03:24+00:00,member_5
1,UgwY2NHHcdFkEv_cggl4AaABAg,ngc324I-IWM,UgwY2NHHcdFkEv_cggl4AaABAg,None,False,UCzLs8wFctJewJjm9VgzQe1Q,@LatencyLate,Just get a Jason stathom head sculpt and use a...,Just get a Jason stathom head sculpt and use a...,0,2026-04-10 06:02:32+00:00,2026-04-10 06:02:32+00:00,0.0,figurines review,2026-04-10 06:04:11+00:00,member_5
2,UgyChHM8gTyofDJZcYV4AaABAg.AVPJ_Byw7Y7AVPgc55vXiN,HPy42rjmYlM,UgyChHM8gTyofDJZcYV4AaABAg,UgyChHM8gTyofDJZcYV4AaABAg,True,UCHKINgvOgV7FT19ByPhfUdQ,@KingMATTtheSuperior,Blanket rolls and bags would make a nice addit...,Blanket rolls and bags would make a nice addit...,0,2026-04-10 06:02:29+00:00,2026-04-10 06:02:29+00:00,NaN,figurines review,2026-04-10 06:04:07+00:00,member_5
3,Ugx0WD-ByYecy8A8xyx4AaABAg,JiKza7DkQ98,Ugx0WD-ByYecy8A8xyx4AaABAg,None,False,UClMkwbl4Sd1szWtIcW8IMpA,@Dolls.and.Ponies,Giant Barbie😂 her face looks like the Karl fac...,Giant Barbie😂 her face looks like the Karl fac...,0,2026-04-10 06:01:14+00:00,2026-04-10 06:01:14+00:00,0.0,collectibles unboxing,2026-04-10 06:02:17+00:00,member_5
4,UgxEUqssJ5xykb-8T_t4AaABAg,khHRah2q1zc,UgxEUqssJ5xykb-8T_t4AaABAg,None,False,UCJM5Ly2wM89MXDgPsm04uUw,@SubhamPal-iq3xi,Yesyes❤❤❤😊😊😊🎉🎉🎉,Yesyes❤❤❤😊😊😊🎉🎉🎉,1,2026-04-10 06:01:03+00:00,2026-04-10 06:01:03+00:00,0.0,makeup kit unboxing,2026-04-10 06:04:39+00:00,member_5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133105,Ugx0GkJFURVpW38GVY94AaABAg,iLLZyHkSaIQ,Ugx0GkJFURVpW38GVY94AaABAg,None,False,UCE974O2-P4Fj6wo60P5Wyvw,@AndersAssarsjö,"Hi Nata, this was interesting to see how much ...","Hi Nata, this was interesting to see how much ...",1,2026-03-02 13:50:37+00:00,2026-03-02 13:50:37+00:00,4.0,camping gear travel setup,2026-04-02 14:26:05+00:00,member_5
133106,UgwTXxgcemNhNdBXm494AaABAg,Ow54NoH-Xmo,UgwTXxgcemNhNdBXm494AaABAg,None,False,UCio4qZwwtQ44_nXcXy-03Vg,@SarasRealWorldReviews,Product Link: https://amzn.to/4aJWTXx,Product Link: https://amzn.to/4aJWTXx,1,2026-02-17 16:20:51+00:00,2026-02-17 16:20:51+00:00,0.0,travel gadgets packing,2026-04-02 13:32:36+00:00,leader
133107,UgwGfA-0jmL1cP0lshB4AaABAg,SObewZBpTms,UgwGfA-0jmL1cP0lshB4AaABAg,None,False,UCtBGFxhujmb4wyEJBXhEABw,@MarcusFerguson16,"This is my B-Camera Travel Bag, not a “perfect...","This is my B-Camera Travel Bag, not a “perfect...",0,2026-01-23 22:43:04+00:00,2026-01-23 22:43:04+00:00,0.0,photography gear travel setup,2026-04-10 05:12:39+00:00,member_1
133108,Ugy4v8U6b79YStq6n154AaABAg,rFTItY-EFPA,Ugy4v8U6b79YStq6n154AaABAg,None,False,UCKC0i7C82FrTe15Q8x5mYZg,@NitecoreStore,"Is this your first EDC flashlight, or are you ...","Is this your first EDC flashlight, or are you ...",2,2025-12-30 19:08:40+00:00,2025-12-30 19:08:40+00:00,1.0,drone daily carry,2026-04-02 13:53:04+00:00,member_2


,run_tag,team_member,search_keyword,published_after,candidate_video_ids,videos_fetched,comments_fetched,run_started_at_utc,logged_at_utc
0,20260402_212816,leader,desk setup tech gadgets,2026-03-03T13:28:16Z,100,100,405,2026-04-02T13:28:16Z,2026-04-02T13:29:00Z
1,20260402_212816,leader,gaming setup accessories,2026-03-03T13:28:16Z,100,100,883,2026-04-02T13:28:16Z,2026-04-02T13:29:35Z
2,20260402_212816,leader,what's in my bag tech,2026-03-03T13:28:16Z,83,83,1428,2026-04-02T13:28:16Z,2026-04-02T13:30:13Z
3,20260402_212816,leader,college tech essentials,2026-03-03T13:28:16Z,2,2,5,2026-04-02T13:28:16Z,2026-04-02T13:30:16Z
4,20260402_212816,leader,camera bag setup,2026-03-03T13:28:16Z,68,68,798,2026-04-02T13:28:16Z,2026-04-02T13:30:40Z
...,...,...,...,...,...,...,...,...,...
611,20260410_135216,member_5,medical devices review,2026-03-11T05:52:16Z,0,0,0,2026-04-10T05:52:16Z,2026-04-10T06:04:57Z
612,20260410_135216,member_5,medical devices travel setup,2026-03-11T05:52:16Z,0,0,0,2026-04-10T05:52:16Z,2026-04-10T06:04:57Z
613,20260410_135216,member_5,medical devices bag setup,2026-03-11T05:52:16Z,0,0,0,2026-04-10T05:52:16Z,2026-04-10T06:04:57Z
614,20260410_135216,member_5,how to protect medical devices,2026-03-11T05:52:16Z,0,0,0,2026-04-10T05:52:16Z,2026-04-10T06:04:57Z


In [11]:
# 保存 master 数据
videos_master_new.to_parquet(VIDEOS_MASTER_FILE, index=False)
comments_master_new.to_parquet(COMMENTS_MASTER_FILE, index=False)
runs_master_new.to_parquet(RUNS_MASTER_FILE, index=False)

print("✅ Master data saved!")

✅ Master data saved!


In [12]:
# 保存为csv
videos_master_new.to_csv(MASTER_DIR / "videos_master.csv", index=False)
comments_master_new.to_csv(MASTER_DIR / "comments_master.csv", index=False)
runs_master_new.to_csv(MASTER_DIR / "runs_master.csv", index=False)

print("✅ 成功保存csv!")


✅ 成功保存csv!


In [15]:
comments_master_new["text_original"]

0                                                Cute❤❤❤❤❤❤
1         Just get a Jason stathom head sculpt and use a...
2         Blanket rolls and bags would make a nice addit...
3         Giant Barbie😂 her face looks like the Karl fac...
4                                           Yesyes❤❤❤😊😊😊🎉🎉🎉
                                ...                        
133105    Hi Nata, this was interesting to see how much ...
133106                Product Link: https://amzn.to/4aJWTXx
133107    This is my B-Camera Travel Bag, not a “perfect...
133108    Is this your first EDC flashlight, or are you ...
133109       Pick it up on Amazon: https://geni.us/cahM #AD
Name: text_original, Length: 133110, dtype: object